# CELL 1: Imports & Config

In [1]:
import os, gc
import numpy as np
import pandas as pd
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
from PIL import Image

# Memory cleanup
torch.cuda.empty_cache()
gc.collect()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


Device: cuda


# CELL 2: Load PAD-UFES-20 Metadata

In [3]:
DATA_DIR = "/kaggle/input/datasets/maxjen/pad-ufes-20/PAD-UFES-20"
IMG_DIR = os.path.join(DATA_DIR, "Dataset")

df = pd.read_csv(os.path.join(DATA_DIR, "metadata.csv"))

print(df.columns)  # sanity check

# ✅ USE img_id (NOT image)
df["image_path"] = df["img_id"].apply(
    lambda x: os.path.join(IMG_DIR, x)
)

# keep only valid images
df = df[df["image_path"].apply(os.path.exists)].reset_index(drop=True)

print("Total images:", len(df))
df.head()

Index(['patient_id', 'lesion_id', 'smoke', 'drink', 'background_father',
       'background_mother', 'age', 'pesticide', 'gender',
       'skin_cancer_history', 'cancer_history', 'has_piped_water',
       'has_sewage_system', 'fitspatrick', 'region', 'diameter_1',
       'diameter_2', 'diagnostic', 'itch', 'grew', 'hurt', 'changed', 'bleed',
       'elevation', 'img_id', 'biopsed'],
      dtype='object')
Total images: 2298


,patient_id,lesion_id,smoke,drink,background_father,background_mother,age,pesticide,gender,skin_cancer_history,...,diagnostic,itch,grew,hurt,changed,bleed,elevation,img_id,biopsed,image_path
0,PAT_1516,1765,NaN,NaN,NaN,NaN,8,NaN,NaN,NaN,...,NEV,False,False,False,False,False,False,PAT_1516_1765_530.png,False,/kaggle/input/datasets/maxjen/pad-ufes-20/PAD-...
1,PAT_46,881,False,False,POMERANIA,POMERANIA,55,False,FEMALE,True,...,BCC,True,True,False,True,True,True,PAT_46_881_939.png,True,/kaggle/input/datasets/maxjen/pad-ufes-20/PAD-...
2,PAT_1545,1867,NaN,NaN,NaN,NaN,77,NaN,NaN,NaN,...,ACK,True,False,False,False,False,False,PAT_1545_1867_547.png,False,/kaggle/input/datasets/maxjen/pad-ufes-20/PAD-...
3,PAT_1989,4061,NaN,NaN,NaN,NaN,75,NaN,NaN,NaN,...,ACK,True,False,False,False,False,False,PAT_1989_4061_934.png,False,/kaggle/input/datasets/maxjen/pad-ufes-20/PAD-...
4,PAT_684,1302,False,True,POMERANIA,POMERANIA,79,False,MALE,True,...,BCC,True,True,False,False,True,True,PAT_684_1302_588.png,True,/kaggle/input/datasets/maxjen/pad-ufes-20/PAD-...


# CELL 3: Encode Labels & Metadata

In [5]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

# ---- LABELS ----
le = LabelEncoder()
df["label"] = le.fit_transform(df["diagnostic"])
num_classes = len(le.classes_)

print("Classes:", le.classes_)
print("Num classes:", num_classes)


Classes: ['ACK' 'BCC' 'MEL' 'NEV' 'SCC' 'SEK']
Num classes: 6


# CELL 4: Train / Val Split & Class Weights

In [7]:
from collections import Counter

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

class_counts = Counter(train_df["label"])

class_weights = torch.tensor(
    [1.0 / class_counts[i] for i in range(num_classes)],
    dtype=torch.float
).to(device)

criterion_ce = nn.CrossEntropyLoss(weight=class_weights)

print("Train:", len(train_df), "Val:", len(val_df))


Train: 1838 Val: 460


# CELL 5: Multi-Scale Dataset (IDENTICAL LOGIC)

In [8]:
class PADUFESMultiScaleDataset(Dataset):
    def __init__(self, df, transform, img_size=224):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def _center_crop(self, img):
        w, h = img.size
        m = min(w, h)
        return img.crop(((w-m)//2, (h-m)//2, (w+m)//2, (h+m)//2))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["image_path"]).convert("RGB")

        full = img.resize((self.img_size, self.img_size))
        center = self._center_crop(img).resize((self.img_size, self.img_size))
        rand = T.RandomResizedCrop(self.img_size, scale=(0.7, 1.0))(img)

        views = [self.transform(v) for v in [full, center, rand]]
        images = torch.stack(views)

        label = torch.tensor(row["label"]).long()
        meta = torch.tensor(row["meta"])

        return images, meta, label


# CELL 6: Transforms & Loaders

In [9]:
train_tf = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(20),
    T.ColorJitter(0.2, 0.2, 0.2),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_tf = T.Compose([
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

train_ds = PADUFESMultiScaleDataset(train_df, train_tf)
val_ds   = PADUFESMultiScaleDataset(val_df, val_tf)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2)


# CELL 7: Metadata Encoder (UNCHANGED)

In [10]:
class MetaEncoder(nn.Module):
    def __init__(self, in_dim=3, out_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64),
            nn.ReLU(),
            nn.Linear(64, out_dim)
        )

    def forward(self, x):
        return self.net(x)


# CELL 8: Image Encoder (UNCHANGED)

In [11]:
class ImageEncoder(nn.Module):
    def __init__(self, embed_dim=512):
        super().__init__()
        base = efficientnet_b3(weights=EfficientNet_B3_Weights.IMAGENET1K_V1)
        self.features = base.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Linear(1536, embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=8, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

    def forward(self, x):
        B, S, C, H, W = x.shape
        feats = []

        for i in range(S):
            f = self.features(x[:, i])
            f = self.pool(f).flatten(1)
            f = self.proj(f)
            feats.append(f)

        seq = torch.stack(feats, dim=1)
        trans = self.transformer(seq)

        return trans.mean(dim=1), feats


# CELL 9: Full Model + Consistency Loss

In [12]:
class FullModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.img_enc = ImageEncoder()
        self.meta_enc = MetaEncoder()

        self.classifier = nn.Sequential(
            nn.Linear(512 + 128, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, images, meta):
        img_embed, feats = self.img_enc(images)
        meta_embed = self.meta_enc(meta)
        fused = torch.cat([img_embed, meta_embed], dim=1)
        logits = self.classifier(fused)
        return logits, feats


def feature_consistency_loss(feats):
    ref = F.normalize(feats[0], dim=1)
    loss = 0
    for f in feats[1:]:
        loss += F.mse_loss(ref, F.normalize(f, dim=1))
    return loss / (len(feats) - 1)


# CELL 10: Training & Validation

In [15]:
model = FullModel(num_classes).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

lambda_cons = 0.3
epochs = 40

for ep in range(epochs):

    # ---- TRAIN ----
    model.train()
    train_correct, train_total = 0, 0

    for imgs, meta, lbl in train_loader:
        imgs, meta, lbl = imgs.to(device), meta.to(device), lbl.to(device)

        optimizer.zero_grad()
        logits, feats = model(imgs, meta)

        loss = criterion_ce(logits, lbl) + lambda_cons * feature_consistency_loss(feats)
        loss.backward()
        optimizer.step()

        train_correct += (logits.argmax(1) == lbl).sum().item()
        train_total += lbl.size(0)

    train_acc = train_correct / train_total

    # ---- VALIDATION ----
    model.eval()
    val_correct, val_total = 0, 0
    all_probs, all_labels = [], []

    with torch.no_grad():
        for imgs, meta, lbl in val_loader:
            imgs, meta, lbl = imgs.to(device), meta.to(device), lbl.to(device)
            logits, _ = model(imgs, meta)
            probs = torch.softmax(logits, dim=1)

            val_correct += (logits.argmax(1) == lbl).sum().item()
            val_total += lbl.size(0)

            all_probs.append(probs.cpu())
            all_labels.append(lbl.cpu())

    val_acc = val_correct / val_total

    all_probs = torch.cat(all_probs).numpy()
    all_labels = torch.cat(all_labels).numpy()

    roc_auc = roc_auc_score(
        all_labels, all_probs, multi_class="ovr", average="macro"
    )

    print(
        f"Epoch {ep+1:02d} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Macro AUC: {roc_auc:.4f}"
    )


KeyError: Caught KeyError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pandas/core/indexes/base.py", line 3805, in get_loc
    return self._engine.get_loc(casted_key)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/hashtable_class_helper.pxi", line 7081, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas/_libs/hashtable_class_helper.pxi", line 7089, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'meta'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 349, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  Fil

In [14]:
META_COLS = [
    "age",
    "gender",
    "smoke",
    "drink",
    "pesticide",
    "skin_cancer_history",
    "cancer_history",
    "itch",
    "grew",
    "hurt",
    "changed",
    "bleed",
    "elevation",
    "fitspatrick",
]

# Encode categorical columns
for col in META_COLS:
    if df[col].dtype == "object":
        df[col] = df[col].astype("category").cat.codes

# Fill missing values
df[META_COLS] = df[META_COLS].fillna(0)

# Create meta vector
df["meta"] = df[META_COLS].values.tolist()

META_DIM = len(META_COLS)


In [18]:
# ===============================
# PAD-UFES-20 Training with Multi-Scale CNN + Transformer + Metadata
# ===============================

import os, gc
import numpy as np
import pandas as pd
from collections import Counter
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# -------------------------------
# Config
# -------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.empty_cache()
gc.collect()

DATA_DIR = "/kaggle/input/datasets/maxjen/pad-ufes-20/PAD-UFES-20"
IMG_DIR = os.path.join(DATA_DIR, "Dataset")
META_FILE = os.path.join(DATA_DIR, "metadata.csv")

# -------------------------------
# Load metadata
# -------------------------------
df = pd.read_csv(META_FILE)

# Map image paths
df["image_path"] = df["img_id"].apply(lambda x: os.path.join(IMG_DIR, x))

# Encode label (diagnostic)
le = LabelEncoder()
df["label"] = le.fit_transform(df["diagnostic"])
num_classes = len(le.classes_)
print("Classes:", le.classes_)

# Metadata: age, gender, fitspatrick, region
df["age"] = df["age"].fillna(df["age"].median())
df["gender"] = df["gender"].map({"male":0,"female":1}).fillna(0)
df["fitspatrick"] = df["fitspatrick"].fillna(df["fitspatrick"].mode()[0])
df["region"] = df["region"].fillna(df["region"].mode()[0])

fitspatrick_enc = LabelEncoder()
region_enc = LabelEncoder()
df["fitspatrick"] = fitspatrick_enc.fit_transform(df["fitspatrick"])
df["region"] = region_enc.fit_transform(df["region"])

scaler = StandardScaler()
df["age_norm"] = scaler.fit_transform(df[["age"]])

df["meta"] = df.apply(
    lambda r: np.array([r["age_norm"], r["gender"], r["fitspatrick"], r["region"]], dtype=np.float32),
    axis=1
)

# -------------------------------
# Train/Val split
# -------------------------------
train_df, val_df = train_test_split(
    df, test_size=0.2, stratify=df["label"], random_state=SEED
)

class_counts = Counter(train_df["label"])
class_weights = torch.tensor([1.0/class_counts[i] for i in range(num_classes)], dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

# -------------------------------
# Dataset
# -------------------------------
class PADUFESDataset(Dataset):
    def __init__(self, df, transform, img_size=224):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def _center_crop(self, img):
        w, h = img.size
        min_side = min(w, h)
        left = (w - min_side)//2
        top = (h - min_side)//2
        return img.crop((left, top, left+min_side, top+min_side))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["image_path"]).convert("RGB")

        full = img.resize((self.img_size,self.img_size))
        center = self._center_crop(img).resize((self.img_size,self.img_size))
        rand = T.RandomResizedCrop(self.img_size, scale=(0.7,1.0))(img)

        views = [full, center, rand]
        views = [self.transform(v) for v in views]
        images = torch.stack(views)  # [3,C,H,W]

        label = torch.tensor(row["label"]).long()
        meta = torch.tensor(row["meta"])
        return images, meta, label

# -------------------------------
# Transforms and loaders
# -------------------------------
train_tf = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(20),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_tf = T.Compose([
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

train_loader = DataLoader(PADUFESDataset(train_df, train_tf), batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(PADUFESDataset(val_df, val_tf), batch_size=8, shuffle=False, num_workers=2)

# -------------------------------
# Model
# -------------------------------
class MetaEncoder(nn.Module):
    def __init__(self, in_dim=4, out_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim,64),
            nn.ReLU(),
            nn.Linear(64,out_dim)
        )
    def forward(self, x):
        return self.net(x)

# ===============================
# Image Encoder (CNN + Transformer) same as your original
# ===============================
class ImageEncoder(nn.Module):
    def __init__(self, embed_dim=512):
        super().__init__()
        base = efficientnet_b3(weights=EfficientNet_B3_Weights.IMAGENET1K_V1)
        self.features = base.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Linear(1536, embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=8, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

    def forward(self, x):
        B,S,C,H,W = x.shape
        feats = []
        for i in range(S):
            f = self.features(x[:,i])
            f = self.pool(f).flatten(1)
            f = self.proj(f)
            feats.append(f)
        seq = torch.stack(feats, dim=1)
        trans = self.transformer(seq)
        return trans.mean(dim=1), feats

class FullModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.img_enc = ImageEncoder()
        self.meta_enc = MetaEncoder()
        self.classifier = nn.Sequential(
            nn.Linear(512+128,256),
            nn.ReLU(),
            nn.Linear(256,num_classes)
        )

    def forward(self, images, meta):
        img_embed, feats = self.img_enc(images)
        meta_embed = self.meta_enc(meta)
        fused = torch.cat([img_embed, meta_embed], dim=1)
        logits = self.classifier(fused)
        return logits, feats

# -------------------------------
# Feature consistency loss
# -------------------------------
def consistency_loss(feats):
    ref = F.normalize(feats[0], dim=1)
    loss = 0
    for f in feats[1:]:
        loss += F.mse_loss(ref, F.normalize(f, dim=1))
    return loss / (len(feats)-1)

# -------------------------------
# Training
# -------------------------------
model = FullModel(num_classes).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
lambda_cons = 0.3
epochs = 40

mel_idx = list(le.classes_).index("melanoma") if "melanoma" in le.classes_ else 0

for ep in range(epochs):
    # TRAIN
    model.train()
    train_total, train_correct = 0,0
    for imgs, meta, lbl in train_loader:
        imgs, meta, lbl = imgs.to(device), meta.to(device), lbl.to(device)
        opt.zero_grad()
        logits, feats = model(imgs, meta)
        loss = criterion(logits, lbl) + lambda_cons * consistency_loss(feats)
        loss.backward()
        opt.step()
        train_correct += (logits.argmax(1)==lbl).sum().item()
        train_total += lbl.size(0)
    train_acc = train_correct/train_total

    # VALIDATION
    model.eval()
    val_total, val_correct = 0,0
    all_probs, all_labels = [],[]
    with torch.no_grad():
        for imgs, meta, lbl in val_loader:
            imgs, meta, lbl = imgs.to(device), meta.to(device), lbl.to(device)
            logits, _ = model(imgs, meta)
            probs = torch.softmax(logits, dim=1)
            all_probs.append(probs.cpu())
            all_labels.append(lbl.cpu())
            val_correct += (logits.argmax(1)==lbl).sum().item()
            val_total += lbl.size(0)
    val_acc = val_correct/val_total

    probs = torch.cat(all_probs).numpy()
    labels = torch.cat(all_labels).numpy()
    macro_auc = roc_auc_score(labels, probs, multi_class="ovr", average="macro")
    mel_auc = roc_auc_score((labels==mel_idx).astype(int), probs[:,mel_idx])

    print(f"Epoch {ep+1:02d} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Macro AUC: {macro_auc:.4f} | Mel AUC: {mel_auc:.4f}")


Classes: ['ACK' 'BCC' 'MEL' 'NEV' 'SCC' 'SEK']
Epoch 01 | Train Acc: 0.5250 | Val Acc: 0.4978 | Macro AUC: 0.8791 | Mel AUC: 0.8818
Epoch 02 | Train Acc: 0.6257 | Val Acc: 0.5717 | Macro AUC: 0.8930 | Mel AUC: 0.8960
Epoch 03 | Train Acc: 0.6627 | Val Acc: 0.6457 | Macro AUC: 0.9010 | Mel AUC: 0.8895
Epoch 04 | Train Acc: 0.7203 | Val Acc: 0.5326 | Macro AUC: 0.9132 | Mel AUC: 0.8889
Epoch 05 | Train Acc: 0.7443 | Val Acc: 0.6261 | Macro AUC: 0.9129 | Mel AUC: 0.8712
Epoch 06 | Train Acc: 0.7671 | Val Acc: 0.6109 | Macro AUC: 0.9161 | Mel AUC: 0.9064
Epoch 07 | Train Acc: 0.7802 | Val Acc: 0.6587 | Macro AUC: 0.9193 | Mel AUC: 0.9006
Epoch 08 | Train Acc: 0.8177 | Val Acc: 0.6457 | Macro AUC: 0.9155 | Mel AUC: 0.9125
Epoch 09 | Train Acc: 0.8237 | Val Acc: 0.6630 | Macro AUC: 0.9208 | Mel AUC: 0.9145
Epoch 10 | Train Acc: 0.8558 | Val Acc: 0.6696 | Macro AUC: 0.9146 | Mel AUC: 0.9147
Epoch 11 | Train Acc: 0.8721 | Val Acc: 0.6500 | Macro AUC: 0.9103 | Mel AUC: 0.9044
Epoch 12 | Train A

KeyboardInterrupt: 

# Stronger data augmentation (RandomResizedCrop + Random Erasing + ColorJitter)

Dropout in metadata & classifier

Label smoothing

AdamW optimizer with weight decay

Cosine Annealing LR scheduler

Feature consistency as before

Here’s a complete modified training script for PAD-UFES-20

In [20]:
# ===============================
# PAD-UFES-20 Training (Reduced Overfitting)
# ===============================

import os, gc
import numpy as np
import pandas as pd
from collections import Counter
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# -------------------------------
# Config
# -------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.empty_cache()
gc.collect()

DATA_DIR = "/kaggle/input/datasets/maxjen/pad-ufes-20/PAD-UFES-20"
IMG_DIR = os.path.join(DATA_DIR, "Dataset")
META_FILE = os.path.join(DATA_DIR, "metadata.csv")

# -------------------------------
# Load metadata
# -------------------------------
df = pd.read_csv(META_FILE)
df["image_path"] = df["img_id"].apply(lambda x: os.path.join(IMG_DIR, x))

le = LabelEncoder()
df["label"] = le.fit_transform(df["diagnostic"])
num_classes = len(le.classes_)
print("Classes:", le.classes_)

# Metadata preprocessing
df["age"] = df["age"].fillna(df["age"].median())
df["gender"] = df["gender"].map({"male":0,"female":1}).fillna(0)
df["fitspatrick"] = df["fitspatrick"].fillna(df["fitspatrick"].mode()[0])
df["region"] = df["region"].fillna(df["region"].mode()[0])

fitspatrick_enc = LabelEncoder()
region_enc = LabelEncoder()
df["fitspatrick"] = fitspatrick_enc.fit_transform(df["fitspatrick"])
df["region"] = region_enc.fit_transform(df["region"])

scaler = StandardScaler()
df["age_norm"] = scaler.fit_transform(df[["age"]])

df["meta"] = df.apply(
    lambda r: np.array([r["age_norm"], r["gender"], r["fitspatrick"], r["region"]], dtype=np.float32),
    axis=1
)

# -------------------------------
# Train/Val split
# -------------------------------
train_df, val_df = train_test_split(
    df, test_size=0.2, stratify=df["label"], random_state=SEED
)
class_counts = Counter(train_df["label"])
class_weights = torch.tensor([1.0/class_counts[i] for i in range(num_classes)], dtype=torch.float).to(device)

# -------------------------------
# Dataset
# -------------------------------
class PADUFESDataset(Dataset):
    def __init__(self, df, transform, img_size=224):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def _center_crop(self, img):
        w,h = img.size
        min_side = min(w,h)
        left = (w-min_side)//2
        top = (h-min_side)//2
        return img.crop((left, top, left+min_side, top+min_side))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["image_path"]).convert("RGB")

        full = img.resize((self.img_size,self.img_size))
        center = self._center_crop(img).resize((self.img_size,self.img_size))
        rand = T.RandomResizedCrop(self.img_size, scale=(0.7,1.0))(img)

        views = [full, center, rand]
        views = [self.transform(v) for v in views]
        images = torch.stack(views)  # [3,C,H,W]

        label = torch.tensor(row["label"]).long()
        meta = torch.tensor(row["meta"])
        return images, meta, label

# -------------------------------
# Strong Transforms
# -------------------------------
train_tf = T.Compose([
    # PIL-based transforms
    T.RandomResizedCrop(224, scale=(0.6,1.0)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(30),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    
    # convert to tensor
    T.ToTensor(),
    
    # tensor-based transforms
    T.RandomErasing(p=0.3, scale=(0.02,0.15)),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])


train_loader = DataLoader(PADUFESDataset(train_df, train_tf), batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(PADUFESDataset(val_df, val_tf), batch_size=8, shuffle=False, num_workers=2)

# -------------------------------
# Model
# -------------------------------
class MetaEncoder(nn.Module):
    def __init__(self, in_dim=4, out_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim,64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64,out_dim)
        )
    def forward(self, x):
        return self.net(x)

class ImageEncoder(nn.Module):
    def __init__(self, embed_dim=512):
        super().__init__()
        base = efficientnet_b3(weights=EfficientNet_B3_Weights.IMAGENET1K_V1)
        self.features = base.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Linear(1536, embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=8, dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

    def forward(self, x):
        B,S,C,H,W = x.shape
        feats = []
        for i in range(S):
            f = self.features(x[:,i])
            f = self.pool(f).flatten(1)
            f = self.proj(f)
            feats.append(f)
        seq = torch.stack(feats, dim=1)
        trans = self.transformer(seq)
        return trans.mean(dim=1), feats

class FullModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.img_enc = ImageEncoder()
        self.meta_enc = MetaEncoder()
        self.classifier = nn.Sequential(
            nn.Linear(512+128,256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256,num_classes)
        )

    def forward(self, images, meta):
        img_embed, feats = self.img_enc(images)
        meta_embed = self.meta_enc(meta)
        fused = torch.cat([img_embed, meta_embed], dim=1)
        logits = self.classifier(fused)
        return logits, feats

# -------------------------------
# Feature consistency loss
# -------------------------------
def consistency_loss(feats):
    ref = F.normalize(feats[0], dim=1)
    loss = 0
    for f in feats[1:]:
        loss += F.mse_loss(ref, F.normalize(f, dim=1))
    return loss / (len(feats)-1)

# -------------------------------
# Optimizer, Scheduler, Criterion
# -------------------------------
model = FullModel(num_classes).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
lambda_cons = 0.3
epochs = 40
mel_idx = list(le.classes_).index("MEL") if "MEL" in le.classes_ else 0

# -------------------------------
# Training Loop
# -------------------------------
for ep in range(epochs):
    # TRAIN
    model.train()
    train_total, train_correct = 0,0
    for imgs, meta, lbl in train_loader:
        imgs, meta, lbl = imgs.to(device), meta.to(device), lbl.to(device)
        optimizer.zero_grad()
        logits, feats = model(imgs, meta)
        loss = criterion(logits, lbl) + lambda_cons * consistency_loss(feats)
        loss.backward()
        optimizer.step()
        train_correct += (logits.argmax(1)==lbl).sum().item()
        train_total += lbl.size(0)
    train_acc = train_correct/train_total

    # VALIDATION
    model.eval()
    val_total, val_correct = 0,0
    all_probs, all_labels = [],[]
    with torch.no_grad():
        for imgs, meta, lbl in val_loader:
            imgs, meta, lbl = imgs.to(device), meta.to(device), lbl.to(device)
            logits, _ = model(imgs, meta)
            probs = torch.softmax(logits, dim=1)
            all_probs.append(probs.cpu())
            all_labels.append(lbl.cpu())
            val_correct += (logits.argmax(1)==lbl).sum().item()
            val_total += lbl.size(0)
    val_acc = val_correct/val_total

    probs = torch.cat(all_probs).numpy()
    labels = torch.cat(all_labels).numpy()
    macro_auc = roc_auc_score(labels, probs, multi_class="ovr", average="macro")
    mel_auc = roc_auc_score((labels==mel_idx).astype(int), probs[:,mel_idx])

    print(f"Epoch {ep+1:02d} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Macro AUC: {macro_auc:.4f} | Mel AUC: {mel_auc:.4f}")
    scheduler.step()


Classes: ['ACK' 'BCC' 'MEL' 'NEV' 'SCC' 'SEK']
Epoch 01 | Train Acc: 0.4113 | Val Acc: 0.6109 | Macro AUC: 0.8822 | Mel AUC: 0.9500
Epoch 02 | Train Acc: 0.5392 | Val Acc: 0.5587 | Macro AUC: 0.8565 | Mel AUC: 0.8173
Epoch 03 | Train Acc: 0.5979 | Val Acc: 0.6478 | Macro AUC: 0.9174 | Mel AUC: 0.9940
Epoch 04 | Train Acc: 0.6453 | Val Acc: 0.5935 | Macro AUC: 0.8960 | Mel AUC: 0.9104
Epoch 05 | Train Acc: 0.6545 | Val Acc: 0.6065 | Macro AUC: 0.8900 | Mel AUC: 0.8940
Epoch 06 | Train Acc: 0.7160 | Val Acc: 0.6217 | Macro AUC: 0.8792 | Mel AUC: 0.8809
Epoch 07 | Train Acc: 0.7193 | Val Acc: 0.6674 | Macro AUC: 0.9018 | Mel AUC: 0.8973
Epoch 08 | Train Acc: 0.7530 | Val Acc: 0.6043 | Macro AUC: 0.8730 | Mel AUC: 0.9009
Epoch 09 | Train Acc: 0.7617 | Val Acc: 0.6413 | Macro AUC: 0.8796 | Mel AUC: 0.7876
Epoch 10 | Train Acc: 0.7976 | Val Acc: 0.6457 | Macro AUC: 0.8926 | Mel AUC: 0.9311
Epoch 11 | Train Acc: 0.7965 | Val Acc: 0.6609 | Macro AUC: 0.8676 | Mel AUC: 0.8191
Epoch 12 | Train A